# Conversation Memory with LangChain
This notebook demonstrates using LangChain's built-in memory classes for conversation management. LangChain provides ready-to-use memory implementations that integrate seamlessly with chat models, chains, and agents, offering advantages like automatic history management, summarization, and vector-based retrieval.

**Advantages of LangChain Memory:**
- **Modern Implementation**: Uses RunnableWithMessageHistory for better integration with LangChain's runnable interface.
- **Customizable History**: Implement BaseChatMessageHistory for tailored memory behavior.
- **Chain Integration**: Easily plug into complex workflows with LangGraph or custom chains.
- **Flexibility**: Supports custom memory types, vector stores for long-term memory, and persistence.
- **Provider Agnostic**: Works with any LangChain-supported LLM.

**Memory types presented in this notebook:**
- **Buffer Memory**: Store and pass all conversation history.
- **Summary Memory**: Periodically summarize the conversation to keep context without full history.
- **Buffer Window Memory**: Keep only the last N messages.
- **Summary Buffer Memory**: Combines buffer and summary - keeps recent messages and summarizes older ones.
- **Persistent Buffer Memory**: Stores conversation history to disk for persistence across sessions.
- **Vector-Based Memory**: Stores messages as embeddings in a vector database and retrieves semantically relevant history.

In [19]:
# Install LangChain if not already installed
# !pip install langchain langchain-ollama langchain-community

from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from langchain_core.messages import AIMessage, HumanMessage, BaseMessage
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import FileChatMessageHistory
from typing import List

# Helper function for summarization (used by Summary and SummaryBuffer)
def summarize_conversation(llm, messages: List[BaseMessage]) -> str:
    conversation = "\n".join([f"{msg.type}: {msg.content}" for msg in messages])
    return llm.invoke(f"Summarize this conversation: {conversation}").content

# Custom Buffer History (used by Buffer and built upon by others)
class BufferHistory(BaseChatMessageHistory):
    def __init__(self):
        self.messages: List[BaseMessage] = []

    def add_messages(self, messages: List[BaseMessage]) -> None:
        self.messages.extend(messages)

    def get_messages(self) -> List[BaseMessage]:
        return self.messages

    def clear(self) -> None:
        self.messages = []

# Persistent Buffer History (builds on BufferHistory with persistence)
class PersistentBufferHistory(BaseChatMessageHistory):
    def __init__(self, file_path: str):
        self.file_path = file_path
        self.buffer_history = BufferHistory()  # Builds on BufferHistory
        self._load()

    def add_messages(self, messages: List[BaseMessage]) -> None:
        self.buffer_history.add_messages(messages)
        self._save()

    def get_messages(self) -> List[BaseMessage]:
        return self.buffer_history.get_messages()

    def clear(self) -> None:
        self.buffer_history.clear()
        self._save()

    def _save(self):
        import json
        data = [{"type": msg.type, "content": msg.content} for msg in self.buffer_history.get_messages()]
        with open(self.file_path, 'w') as f:
            json.dump(data, f)

    def _load(self):
        import json
        try:
            with open(self.file_path, 'r') as f:
                data = json.load(f)
                messages = []
                for item in data:
                    if item["type"] == "human":
                        messages.append(HumanMessage(content=item["content"]))
                    elif item["type"] == "ai":
                        messages.append(AIMessage(content=item["content"]))
                self.buffer_history.messages = messages
        except FileNotFoundError:
            pass

# Choose a model (local Ollama)
llm = ChatOllama(model='llama3.2:1b')

# Create prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

## Buffer Memory
Keeps the entire conversation history and passes it to each LLM call.

In [20]:
# Store for persistence
buffer_store = {}

# Function to get history
def get_buffer_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in buffer_store:
        buffer_store[session_id] = InMemoryChatMessageHistory()
    return buffer_store[session_id]

# Create chain
buffer_chain = prompt | llm

# Create runnable with history
buffer_runnable = RunnableWithMessageHistory(
    buffer_chain,
    get_buffer_history,
    input_messages_key="input",
    history_messages_key="history"
)

# Chat
response1 = buffer_runnable.invoke({"input": "Hello, my name is Alice."}, config={"configurable": {"session_id": "1"}})
print('AI:', response1.content)

response2 = buffer_runnable.invoke({"input": "What is my name?"}, config={"configurable": {"session_id": "1"}})
print('AI:', response2.content)

# Get history
history = get_buffer_history("1")
print('Buffer History:', [msg.content for msg in history.messages])

## Summary Memory
Summarizes the conversation periodically to reduce token usage while maintaining context.

In [21]:
class SummaryHistory(BaseChatMessageHistory):
    def __init__(self, llm):
        self.messages: List[BaseMessage] = []
        self.summary = ""
        self.llm = llm

    def add_messages(self, messages: List[BaseMessage]) -> None:
        self.messages.extend(messages)
        # Summarize when too many messages
        if len(self.messages) > 4:  # Simple threshold
            self.summary = summarize_conversation(self.llm, self.messages)
            self.messages = []  # Clear messages after summarizing

    def get_messages(self) -> List[BaseMessage]:
        if self.summary:
            return [AIMessage(content=f"Summary: {self.summary}")] + self.messages
        return self.messages

    def clear(self) -> None:
        self.messages = []
        self.summary = ""

# Store for persistence
summary_store = {}

# Function to get history
def get_summary_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in summary_store:
        summary_store[session_id] = SummaryHistory(llm)
    return summary_store[session_id]

# Create chain
summary_chain = prompt | llm

# Create runnable with history
summary_runnable = RunnableWithMessageHistory(
    summary_chain,
    get_summary_history,
    input_messages_key="input",
    history_messages_key="history"
)

# Chat
response1 = summary_runnable.invoke({"input": "I like pizza."}, config={"configurable": {"session_id": "2"}})
print('AI:', response1.content)

response2 = summary_runnable.invoke({"input": "What do I like?"}, config={"configurable": {"session_id": "2"}})
print('AI:', response2.content)

response3 = summary_runnable.invoke({"input": "Tell me about my preferences."}, config={"configurable": {"session_id": "2"}})
print('AI:', response3.content)

response4 = summary_runnable.invoke({"input": "Remind me what we talked about."}, config={"configurable": {"session_id": "2"}})
print('AI:', response4.content)

# Get history
history = get_summary_history("2")
print('Summary History:', [msg.content for msg in history.get_messages()])

## Buffer Window Memory
Keeps only the last N messages to limit context length.

In [22]:
class WindowHistory(BaseChatMessageHistory):
    def __init__(self, k: int = 2):
        self.messages: List[BaseMessage] = []
        self.k = k

    def add_messages(self, messages: List[BaseMessage]) -> None:
        self.messages.extend(messages)
        # Keep only last 2k messages (since each interaction is user + AI)
        if len(self.messages) > self.k * 2:
            self.messages = self.messages[-self.k * 2:]

    def get_messages(self) -> List[BaseMessage]:
        return self.messages

    def clear(self) -> None:
        self.messages = []

# Store for persistence
window_store = {}

# Function to get history
def get_window_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in window_store:
        window_store[session_id] = WindowHistory(k=2)
    return window_store[session_id]

# Create chain
window_chain = prompt | llm

# Create runnable with history
window_runnable = RunnableWithMessageHistory(
    window_chain,
    get_window_history,
    input_messages_key="input",
    history_messages_key="history"
)

# Chat
response1 = window_runnable.invoke({"input": "First message."}, config={"configurable": {"session_id": "3"}})
print('AI:', response1.content)

response2 = window_runnable.invoke({"input": "Second message."}, config={"configurable": {"session_id": "3"}})
print('AI:', response2.content)

response3 = window_runnable.invoke({"input": "Third message."}, config={"configurable": {"session_id": "3"}})
print('AI:', response3.content)

response4 = window_runnable.invoke({"input": "Fourth message."}, config={"configurable": {"session_id": "3"}})
print('AI:', response4.content)

response5 = window_runnable.invoke({"input": "What was the first message?"}, config={"configurable": {"session_id": "3"}})
print('AI:', response5.content)

# Get history
history = get_window_history("3")
print('Window History:', [msg.content for msg in history.get_messages()])

## Summary Buffer Memory
Combines buffer and summary: keeps recent messages and summarizes older ones.

In [23]:
class SummaryBufferHistory(BaseChatMessageHistory):
    def __init__(self, llm, max_tokens: int = 100):
        self.buffer_history = BufferHistory()  # Builds on BufferHistory
        self.summary = ""
        self.llm = llm
        self.max_tokens = max_tokens

    def add_messages(self, messages: List[BaseMessage]) -> None:
        self.buffer_history.add_messages(messages)  # Use BufferHistory
        # Estimate tokens (rough approximation)
        total_tokens = sum(len(msg.content.split()) for msg in self.buffer_history.get_messages())
        if total_tokens > self.max_tokens:
            # Summarize older messages
            all_messages = self.buffer_history.get_messages()
            recent = all_messages[-4:]  # Keep last 4
            older = all_messages[:-4]
            if older:
                self.summary = summarize_conversation(self.llm, older)
            # Clear and keep only recent
            self.buffer_history.clear()
            self.buffer_history.add_messages(recent)

    def get_messages(self) -> List[BaseMessage]:
        messages = self.buffer_history.get_messages()
        if self.summary:
            return [AIMessage(content=f"Summary: {self.summary}")] + messages
        return messages

    def clear(self) -> None:
        self.buffer_history.clear()
        self.summary = ""

    @property
    def messages(self) -> List[BaseMessage]:
        return self.get_messages()

# Store for persistence
summary_buffer_store = {}

# Function to get history
def get_summary_buffer_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in summary_buffer_store:
        summary_buffer_store[session_id] = SummaryBufferHistory(llm, max_tokens=100)
    return summary_buffer_store[session_id]

# Create chain
summary_buffer_chain = prompt | llm

# Create runnable with history
summary_buffer_runnable = RunnableWithMessageHistory(
    summary_buffer_chain,
    get_summary_buffer_history,
    input_messages_key="input",
    history_messages_key="history"
)

# Chat with longer conversation
response1 = summary_buffer_runnable.invoke({"input": "Tell me a joke."}, config={"configurable": {"session_id": "4"}})
print('AI:', response1.content)

response2 = summary_buffer_runnable.invoke({"input": "Another one please."}, config={"configurable": {"session_id": "4"}})
print('AI:', response2.content)

response3 = summary_buffer_runnable.invoke({"input": "One more joke."}, config={"configurable": {"session_id": "4"}})
print('AI:', response3.content)

response4 = summary_buffer_runnable.invoke({"input": "Tell me a funny story."}, config={"configurable": {"session_id": "4"}})
print('AI:', response4.content)

response5 = summary_buffer_runnable.invoke({"input": "What jokes have I heard?"}, config={"configurable": {"session_id": "4"}})
print('AI:', response5.content)

# Get history
history = get_summary_buffer_history("4")
print('Summary Buffer History:', [msg.content for msg in history.get_messages()])

## Persistent Buffer Memory
Stores conversation history to disk for persistence across sessions.

In [24]:
# Store for persistence
persistent_store = {}

# Function to get history
def get_persistent_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in persistent_store:
        persistent_store[session_id] = FileChatMessageHistory(f"chat_history_{session_id}.json")
    return persistent_store[session_id]

# Create chain
persistent_chain = prompt | llm

# Create runnable with history
persistent_runnable = RunnableWithMessageHistory(
    persistent_chain,
    get_persistent_history,
    input_messages_key="input",
    history_messages_key="history"
)

# Chat
response1 = persistent_runnable.invoke({"input": "Remember my favorite color is blue."}, config={"configurable": {"session_id": "5"}})
print('AI:', response1.content)

response2 = persistent_runnable.invoke({"input": "What is my favorite color?"}, config={"configurable": {"session_id": "5"}})
print('AI:', response2.content)

print('Persistent History:', [msg.content for msg in history.messages])

## Vector-Based Memory
Stores messages as embeddings in a vector database and retrieves semantically relevant history.

In [25]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.messages import HumanMessage, AIMessage

class VectorChatMessageHistory(BaseChatMessageHistory):
    def __init__(self, vectorstore, k=2):
        self.vectorstore = vectorstore
        self.k = k
        self.messages = []

    def add_messages(self, messages):
        self.messages.extend(messages)
        # Add to vector store
        for msg in messages:
            self.vectorstore.add_texts([msg.content], metadatas=[{"role": msg.type}])

    def get_messages(self):
        if not self.messages:
            return []
        # Retrieve relevant messages
        query = self.messages[-1].content  # Use last message as query
        docs = self.vectorstore.similarity_search(query, k=self.k)
        relevant_contents = [doc.page_content for doc in docs]
        # Return as messages (simplified)
        return [AIMessage(content="Relevant history: " + "; ".join(relevant_contents))]

    def clear(self):
        self.messages = []
        self.vectorstore.delete_collection()

# Setup
embeddings = OpenAIEmbeddings()
vectorstore = Chroma(embedding_function=embeddings, collection_name="chat_memory")

# Store
vector_store = {}

def get_vector_history(session_id: str):
    if session_id not in vector_store:
        vector_store[session_id] = VectorChatMessageHistory(vectorstore, k=2)
    return vector_store[session_id]

# Create runnable
vector_chain = prompt | llm
vector_runnable = RunnableWithMessageHistory(
    vector_chain,
    get_vector_history,
    input_messages_key="input",
    history_messages_key="history"
)

# Chat
response1 = vector_runnable.invoke({"input": "I love programming."}, config={"configurable": {"session_id": "6"}})
print('AI:', response1.content)

response2 = vector_runnable.invoke({"input": "What do I love?"}, config={"configurable": {"session_id": "6"}})
print('AI:', response2.content)

print('Vector History:', [msg.content for msg in get_vector_history("6").get_messages()])